
# ISIN ➜ Yahoo ticker (simple, ISIN-only)

**What this notebook does**  
- Reads your CSV file.  
- Keeps only rows where **Stock ID** (ISIN) is non-empty.  
- Uses Yahoo's public search API (`/v1/finance/search`) to find a ticker **by ISIN**.  
- Writes the result into a new column **`Yahoo ticker`**.  
- Prints how many rows matched vs not matched (with %).  

> **Note:** `yfinance.Ticker("ISIN")` does **not** resolve ISINs to symbols — it just stores whatever
> string you pass. That’s why this notebook uses Yahoo’s search API directly so the ticker is real.


In [ ]:

# 0) Configuration — edit these two paths before running
in_csv  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\uni_pre_cleaned.csv"
out_csv = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\uni_with_yahoo_from_isin.csv"


In [ ]:

# 1) Imports
import time
import json
import pandas as pd
import requests

# small polite delay between HTTP calls
PAUSE = 0.10

# Yahoo search endpoint + headers
YF_SEARCH_URL = "https://query2.finance.yahoo.com/v1/finance/search"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0.0.0 Safari/537.36"),
    "Accept": "application/json,text/plain,*/*",
    "Connection": "keep-alive",
}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)


In [ ]:

# 2) Helper: search Yahoo by ISIN and return the top symbol (or "")
def yahoo_symbol_from_isin(isin: str) -> str:
    if not isin or not isinstance(isin, str):
        return ""
    try:
        params = {"q": isin, "quotesCount": 10, "newsCount": 0, "lang": "en-US", "region": "US"}
        r = SESSION.get(YF_SEARCH_URL, params=params, timeout=15)
        if r.status_code != 200:
            return ""
        data = r.json()
        quotes = data.get("quotes") or []
        if not quotes:
            return ""
        # choose the first "quoteType" in EQUITY/ETF if present, else first result
        for q in quotes:
            if str(q.get("quoteType","")).upper() in {"EQUITY","ETF"} and q.get("symbol"):
                return q["symbol"]
        return quotes[0].get("symbol","") or ""
    except Exception:
        return ""
    finally:
        time.sleep(PAUSE)


In [ ]:

# 3) Read CSV and filter rows with non-empty ISIN in "Stock ID"
NAME_COL = "Name/Kind of Investment Item"
ISIN_COL = "Stock ID"

df = pd.read_csv(in_csv, dtype=str, keep_default_na=False, na_values=[], encoding="cp1252")
work_mask = df.get(ISIN_COL, pd.Series([""]*len(df))).astype(str).str.strip().ne("")
df_work = df.loc[work_mask, [NAME_COL, ISIN_COL]].copy()

print(f"Rows with non-empty ISIN: {len(df_work)} / {len(df)}")


In [ ]:

# 4) Query Yahoo once per unique ISIN (saves time), then map results back
unique_isins = df_work[ISIN_COL].astype(str).str.strip().drop_duplicates().tolist()

symbol_map = {}
done = 0
for isin in unique_isins:
    sym = yahoo_symbol_from_isin(isin)
    symbol_map[isin] = sym
    done += 1
    if done % 200 == 0 or done == len(unique_isins):
        print(f"looked up {done}/{len(unique_isins)} ISINs")

df["Yahoo ticker"] = df.get(ISIN_COL, "").map(lambda x: symbol_map.get(str(x).strip(), ""))


In [ ]:

# 5) Save and summary
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
found = int(df["Yahoo ticker"].astype(str).str.strip().ne("").sum())
eligible = int(df[ISIN_COL].astype(str).str.strip().ne("").sum())
not_found = eligible - found
rate = (found/eligible*100) if eligible else 0.0

print("✅ Saved:", out_csv)
print("---- Summary ----")
print(f"Rows with ISIN (processed): {eligible}")
print(f"Rows that found a ticker:   {found}")
print(f"Rows with no ticker:        {not_found}")
print(f"Hit rate:                   {rate:.2f}%")
